# 4.0 — Modelos Pydantic sobre el grafo Neo4j

Ejercita los esquemas de `src/data/schemas/` alimentados desde el grafo de
farmacovigilancia en **Neo4j** (migración SQLite→Neo4j). El puente ya existe
en `src/data/enrichment.py`; aquí solo lo usamos y validamos.

Si Neo4j no está accesible o autenticado, el notebook **omite** las celdas de
BD con un mensaje y termina sin error.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent  # the notebook lives in notebooks/ -> repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from datetime import date

from neo4j.exceptions import AuthError, ServiceUnavailable

from src.config import neo4j_config
from src.data.enrichment import PharmacovigilanceStore, enrich_drug
from src.data.repository import get_enriched_drug
from src.data.schemas import ATCCode, Drug, Interaction, Med, Patient, SideEffect

from pprint import pprint, pformat

cfg = neo4j_config()
print(f"uri={cfg.uri}  user={cfg.user}  database={cfg.database}")  # never the password

uri=bolt://localhost:7687  user=neo4j  database=neo4j


## Conexión y descubrimiento de un CID

Abre el `PharmacovigilanceStore`. Ante fallo de conexión o autenticación,
`STORE = None` y las celdas siguientes se omiten. Con conexión, busca un CID
de `:Drug` con al menos una arista `HAS_SIDE_EFFECT` (prefiere 33613,
amoxicilina; si no, el primero encontrado).

In [2]:
STORE: PharmacovigilanceStore | None = None
SAMPLE_CID: int | None = None

# Discover a Drug node with at least one side-effect edge, returning the
# whole node so we can show every label and property (not just the CID).
_DISCOVER = (
    "MATCH (d:Drug)-[:HAS_SIDE_EFFECT]->() "
    "WITH DISTINCT d ORDER BY (d.cid = 33613) DESC, d.cid LIMIT 1 "
    "RETURN d.cid AS cid, labels(d) AS labels, properties(d) AS props"
)

try:
    STORE = PharmacovigilanceStore(cfg)
    with STORE._driver.session(database=cfg.database) as _s:
        _rec = _s.run(_DISCOVER).single()
    SAMPLE_CID = _rec["cid"] if _rec else None
    if SAMPLE_CID is None:
        print("Neo4j connected but no HAS_SIDE_EFFECT edges — skipping the DB.")
    else:
        print(f"Connected. Sample CID = {SAMPLE_CID}")
        print(f"  labels: {_rec['labels']}")
        for _key, _value in _rec["props"].items():
            print(f"  {_key}: {_value}")
except (ServiceUnavailable, AuthError) as exc:
    if STORE is not None:
        STORE.close()
    STORE = None
    print(f"Neo4j unavailable ({type(exc).__name__}) — skipping DB cells.")

Neo4j unavailable (AuthError) — skipping DB cells.


## Filas del grafo → modelos validados

`side_effects` / `interactions` devuelven modelos Pydantic ya validados
(provenance obligatoria, código MedDRA con patrón, literales de severidad y
de fuente). Comprobamos el tipo y mostramos los primeros.

In [3]:
if STORE is None or SAMPLE_CID is None:
    print("DB skipped — no effects/interactions to show.")
else:
    effects = STORE.side_effects(SAMPLE_CID)
    interactions = STORE.interactions(SAMPLE_CID)
    assert all(isinstance(e, SideEffect) for e in effects)
    assert all(isinstance(i, Interaction) for i in interactions)
    print(f"CID {SAMPLE_CID}: {len(effects)} effects, {len(interactions)} interactions")
    for e in effects[:10]:
        pprint(e.model_dump())
    for i in interactions[:10]:
        pprint(i.model_dump())

DB skipped — no effects/interactions to show.


## Enriquecer un Drug local (sin red)

Construimos un `Drug` a mano y lo enriquecemos desde el grafo con
`enrich_drug`. Camino principal, sin depender de PubChem.

In [4]:
enriched: Drug | None = None
if STORE is None or SAMPLE_CID is None:
    print("DB skipped — Drug not enriched.")
else:
    base = Drug(cid=SAMPLE_CID, name=f"CID {SAMPLE_CID}")
    enriched = enrich_drug(base, STORE)
    print(f"has_atc={enriched.has_atc}  "
          f"effects={len(enriched.side_effects)}  "
          f"interactions={len(enriched.interactions)}")

DB skipped — Drug not enriched.


## Paciente completo desde el grafo

Envolvemos el `Drug` enriquecido en un `Med` y un `Patient` (misma forma que
el notebook 2.0). Demuestra que los modelos alimentados por el grafo componen
toda la jerarquía del dominio y siguen pasando cada validador.

In [5]:
if enriched is None:
    print("DB skipped — Patient not built.")
else:
    med = Med(
        ATC_code=(enriched.chemical_group if enriched.has_atc else ATCCode(code="J01CA04")),
        name="Medicación de prueba",
        dosage="500 mg",
        frequency="cada 8h",
        start_date=date(2026, 6, 1),
        active_principles=[enriched],
    )
    patient = Patient(
        id=1,
        name="Paciente de prueba",
        age=78,
        birth_date=date(1948, 1, 1),
        number_of_meds=1,
        polymedicated=True,
        diseases=["infección respiratoria"],
        medication=[med],
    )
    print(patient.model_dump())

DB skipped — Patient not built.


## Opcional — PubChem + grafo ⚠️

`get_enriched_drug` resuelve la identidad química vía **PubChem (red)** y le
añade la farmacovigilancia del grafo. No hace falta ejecutarla para validar
el notebook.

In [6]:
if STORE is None or SAMPLE_CID is None:
    print("DB skipped — PubChem not queried.")
else:
    full = get_enriched_drug(SAMPLE_CID, STORE)
    print(full.model_dump() if full else "CID not resolved in PubChem")

DB skipped — PubChem not queried.


## Cierre

In [7]:
if STORE is not None:
    STORE.close()
    print("Store closed.")
else:
    print("No store was open.")

No store was open.
